# 05 Evaluate AE Scores and Thresholds

Compute reconstruction scores for the trained convolutional autoencoder, select thresholds only on validation data, and evaluate the held-out test split with frozen thresholds.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, precision_recall_curve
import tensorflow as tf


I0000 00:00:1781810320.991985 2193642 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781810321.042958 2193642 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1781810321.981047 2193642 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
REPO_ROOT = Path('..').resolve()
DATASET_CONFIGS = {
    'turning': {
        'manifest_path': REPO_ROOT / 'reports' / 'manifests' / 'turning_split_seed42.csv',
        'model_path': REPO_ROOT / 'models' / 'ae_bn16_20260616.keras',
        'positive_label': 'chatter',
    },
    'cnc_machining': {
        'manifest_path': REPO_ROOT / 'reports' / 'manifests' / 'cnc_machining_split_seed42.csv',
        'model_path': REPO_ROOT / 'models' / 'ae_bn16_cnc_machining_seed42.keras',
        'positive_label': 'anomaly',
    },
}
DATASETS_TO_EVALUATE = tuple(DATASET_CONFIGS)
THRESHOLD_DIR = REPO_ROOT / 'reports' / 'thresholds'
SCORE_DIR = REPO_ROOT / 'reports' / 'scores'
TABLE_DIR = REPO_ROOT / 'reports' / 'tables'

IMAGE_SIZE = (150, 100)
N_VER_SEGMENTS = 10
VER_TOP_K = 3

THRESHOLD_DIR.mkdir(parents=True, exist_ok=True)
SCORE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
manifest_frames = []
for dataset_name in DATASETS_TO_EVALUATE:
    config = DATASET_CONFIGS[dataset_name]
    dataset_manifest = pd.read_csv(config['manifest_path'])
    dataset_manifest = dataset_manifest[(dataset_manifest['source_dataset'] == dataset_name) & dataset_manifest['split'].isin(['validation', 'test'])].copy()
    if 'image_path' not in dataset_manifest.columns:
        print(f'Skipping {dataset_name}: manifest has no image_path column. Run notebook 03 first.')
        continue
    dataset_manifest = dataset_manifest[dataset_manifest['image_path'].fillna('').ne('')].copy()
    dataset_manifest['target'] = (dataset_manifest['label'] == config['positive_label']).astype(int)
    manifest_frames.append(dataset_manifest)

manifest = pd.concat(manifest_frames, ignore_index=True) if manifest_frames else pd.DataFrame(columns=['source_dataset', 'split', 'label', 'image_path', 'target'])
manifest.groupby(['source_dataset', 'split', 'label']).size().rename('n').reset_index() if not manifest.empty else manifest


,source_dataset,split,label,n
0,cnc_machining,test,anomaly,481
1,cnc_machining,test,nominal,5144
2,cnc_machining,validation,anomaly,612
3,cnc_machining,validation,nominal,4975
4,turning,test,chatter,27
5,turning,test,no_chatter,48
6,turning,validation,chatter,34
7,turning,validation,no_chatter,70


In [4]:
def load_images(rows: pd.DataFrame, image_size=IMAGE_SIZE) -> tuple[np.ndarray, pd.DataFrame]:
    images = []
    loaded_rows = []
    for _, row in rows.iterrows():
        image_path = REPO_ROOT / row['image_path']
        if not image_path.exists():
            print(f'Missing image, skipping: {image_path}')
            continue
        image = Image.open(image_path).convert('RGB').resize(image_size)
        images.append(np.asarray(image, dtype='float32') / 255.0)
        loaded_rows.append(row)
    if not images:
        raise ValueError('No images loaded. Run notebooks 02 and 03 first.')
    return np.stack(images, axis=0), pd.DataFrame(loaded_rows).reset_index(drop=True)

# Images are loaded per dataset below so each dataset can use its own model.


In [5]:
def vertical_segment_scores(error_map: np.ndarray, n_segments: int, top_k: int) -> tuple[np.ndarray, np.ndarray]:
    width = error_map.shape[2]
    boundaries = np.linspace(0, width, n_segments + 1, dtype=int)
    segment_scores = []
    for left, right in zip(boundaries[:-1], boundaries[1:]):
        if right <= left:
            continue
        segment_scores.append(error_map[:, :, left:right, :].mean(axis=(1, 2, 3)))
    segment_scores = np.stack(segment_scores, axis=1)
    ver_max = segment_scores.max(axis=1)
    top_k = min(top_k, segment_scores.shape[1])
    ver_topk = np.sort(segment_scores, axis=1)[:, -top_k:].mean(axis=1)
    return ver_max, ver_topk

def score_reconstructions(model, images: np.ndarray) -> pd.DataFrame:
    recon = model.predict(images, verbose=0)
    squared_error = (images - recon) ** 2
    absolute_error = np.abs(images - recon)
    ver_max, ver_topk = vertical_segment_scores(squared_error, N_VER_SEGMENTS, VER_TOP_K)
    return pd.DataFrame({
        'global_mse': squared_error.mean(axis=(1, 2, 3)),
        'global_mae': absolute_error.mean(axis=(1, 2, 3)),
        'ver_max': ver_max,
        'ver_topk': ver_topk,
    })


In [6]:
dataset_score_frames = {}

for dataset_name in DATASETS_TO_EVALUATE:
    config = DATASET_CONFIGS[dataset_name]
    model_path = config['model_path']
    if not model_path.exists():
        print(f'Skipping {dataset_name}: model not found at {model_path}')
        continue

    dataset_manifest = manifest[manifest['source_dataset'] == dataset_name]
    if dataset_manifest.empty:
        print(f'Skipping {dataset_name}: no manifest rows with generated images.')
        continue
    if dataset_manifest[dataset_manifest['split'] == 'validation'].empty or dataset_manifest[dataset_manifest['split'] == 'test'].empty:
        print(f'Skipping {dataset_name}: validation or test images are missing.')
        continue

    validation_images, validation_rows = load_images(dataset_manifest[dataset_manifest['split'] == 'validation'])
    test_images, test_rows = load_images(dataset_manifest[dataset_manifest['split'] == 'test'])
    print(dataset_name, 'validation:', validation_images.shape)
    print(dataset_name, 'test:', test_images.shape)

    model = tf.keras.models.load_model(model_path)
    validation_scores = pd.concat([validation_rows.reset_index(drop=True), score_reconstructions(model, validation_images)], axis=1)
    test_scores = pd.concat([test_rows.reset_index(drop=True), score_reconstructions(model, test_images)], axis=1)
    all_scores = pd.concat([validation_scores, test_scores], ignore_index=True)
    scores_path = SCORE_DIR / f'ae_scores_{dataset_name}.csv'
    all_scores.to_csv(scores_path, index=False)
    dataset_score_frames[dataset_name] = all_scores
    print(f'Wrote {scores_path}')

if dataset_score_frames:
    combined_scores = pd.concat(dataset_score_frames.values(), ignore_index=True)
    combined_scores_path = SCORE_DIR / 'ae_scores_all_datasets.csv'
    combined_scores.to_csv(combined_scores_path, index=False)
    print(f'Wrote {combined_scores_path}')


turning validation: (104, 100, 150, 3)
turning test: (75, 100, 150, 3)


E0000 00:00:1781810322.712160 2193642 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/scores/ae_scores_turning.csv


cnc_machining validation: (5587, 100, 150, 3)
cnc_machining test: (5625, 100, 150, 3)


Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/scores/ae_scores_cnc_machining.csv
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/scores/ae_scores_all_datasets.csv


In [7]:
def select_best_f1_threshold(y_true: np.ndarray, scores: np.ndarray) -> dict:
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    if len(thresholds) == 0:
        threshold = float(np.max(scores))
        return {'threshold': threshold, 'validation_f1': 0.0, 'validation_precision': 0.0, 'validation_recall': 0.0}
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best_idx = int(np.nanargmax(f1))
    return {
        'threshold': float(thresholds[best_idx]),
        'validation_f1': float(f1[best_idx]),
        'validation_precision': float(precision[best_idx]),
        'validation_recall': float(recall[best_idx]),
    }

def evaluate_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'pr_auc': float(average_precision_score(y_true, scores)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }


In [8]:
score_columns = ['global_mse', 'global_mae', 'ver_max', 'ver_topk']
all_metric_rows = []

for dataset_name, all_scores in dataset_score_frames.items():
    validation_scores = all_scores[all_scores['split'] == 'validation']
    test_scores = all_scores[all_scores['split'] == 'test']
    y_val = validation_scores['target'].to_numpy()
    y_test = test_scores['target'].to_numpy()
    thresholds = {}
    metric_rows = []

    for score_name in score_columns:
        selected = select_best_f1_threshold(y_val, validation_scores[score_name].to_numpy())
        thresholds[score_name] = selected
        test_metrics = evaluate_at_threshold(y_test, test_scores[score_name].to_numpy(), selected['threshold'])
        metric_rows.append({
            'dataset': dataset_name,
            'method': 'cnn_ae',
            'score': score_name,
            'threshold': selected['threshold'],
            **test_metrics,
        })

    threshold_path = THRESHOLD_DIR / f'ae_thresholds_{dataset_name}.json'
    metrics_path = TABLE_DIR / f'metrics_{dataset_name}_ae.csv'
    with threshold_path.open('w') as f:
        json.dump(thresholds, f, indent=2)
    pd.DataFrame(metric_rows).to_csv(metrics_path, index=False)
    all_metric_rows.extend(metric_rows)
    print(f'Wrote {threshold_path}')
    print(f'Wrote {metrics_path}')

all_metrics = pd.DataFrame(all_metric_rows)
if not all_metrics.empty:
    combined_metrics_path = TABLE_DIR / 'metrics_ae_all_datasets.csv'
    all_metrics.to_csv(combined_metrics_path, index=False)
    print(f'Wrote {combined_metrics_path}')
all_metrics


Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/thresholds/ae_thresholds_turning.json
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/tables/metrics_turning_ae.csv
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/thresholds/ae_thresholds_cnc_machining.json
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/tables/metrics_cnc_machining_ae.csv
Wrote /home/cgawron/git/spectrogram-anomaly-ae/reports/tables/metrics_ae_all_datasets.csv


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,fn,tp
0,turning,cnn_ae,global_mse,0.002134,0.979938,0.931034,0.870968,1.000000,44,4,0,27
1,turning,cnn_ae,global_mae,0.029933,0.993651,0.928571,0.896552,0.962963,45,3,1,26
2,turning,cnn_ae,ver_max,0.003497,0.965831,0.915254,0.843750,1.000000,43,5,0,27
3,turning,cnn_ae,ver_topk,0.002744,0.968217,0.915254,0.843750,1.000000,43,5,0,27
4,cnc_machining,cnn_ae,global_mse,0.000013,0.068937,0.159481,0.086681,0.995842,97,5047,2,479
5,cnc_machining,cnn_ae,global_mae,0.000348,0.069065,0.160630,0.087361,0.995842,140,5004,2,479
6,cnc_machining,cnn_ae,ver_max,0.000028,0.074791,0.158252,0.085971,0.993763,62,5082,3,478
7,cnc_machining,cnn_ae,ver_topk,0.000032,0.071809,0.160270,0.087229,0.985447,184,4960,7,474
